# BERTweet: Climate Change Pro vs Anti Classification

## 1. import and prepare dataset

In [1]:
# check GPU for fine-tuning bertweet
# reference: https://pytorch.org/get-started/locally/
# code assisted by codex 5.5
import torch

print("torch:", torch.__version__)
print("cuda runtime:", torch.version.cuda)
print("cuda available:", torch.cuda.is_available())
print("gpu:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "no cuda")

assert torch.cuda.is_available(), "CUDA is not available. Please select the hw1 kernel with CUDA PyTorch installed."

torch: 2.11.0+cu128
cuda runtime: 12.8
cuda available: True
gpu: NVIDIA GeForce RTX 5080 Laptop GPU


In [2]:
# import dataset
import pandas as pd

DATA_PATH = "twitter_sentiment_data.csv"
df = pd.read_csv(DATA_PATH)

print(df.shape)
df.head()

(43943, 3)


,sentiment,message,tweetid
0,-1,@tiniebeany climate change is an interesting h...,792927353886371840
1,1,RT @NatGeoChannel: Watch #BeforeTheFlood right...,793124211518832641
2,1,Fabulous! Leonardo #DiCaprio's film on #climat...,793124402388832256
3,1,RT @Mick_Fanning: Just watched this amazing do...,793124635873275904
4,2,"RT @cnalive: Pranita Biswasi, a Lutheran from ...",793125156185137153


In [3]:
# keep only pro and anti tweets 
# create a new column "label" for binary classification which is needed for fine-tuning bertweet
# code assisted by codex 5.5
df = df[df["sentiment"].isin([1, -1])].copy()
df["label"] = df["sentiment"].map({-1: 0, 1: 1})

id2label = {0: "Anti", 1: "Pro"}
label2id = {"Anti": 0, "Pro": 1}

print("rows after filtering:", len(df))
display(df["sentiment"].value_counts().sort_index())
df[["message", "sentiment", "label"]].head()

rows after filtering: 26952


sentiment
-1     3990
 1    22962
Name: count, dtype: int64

,message,sentiment,label
0,@tiniebeany climate change is an interesting h...,-1,0
1,RT @NatGeoChannel: Watch #BeforeTheFlood right...,1,1
2,Fabulous! Leonardo #DiCaprio's film on #climat...,1,1
3,RT @Mick_Fanning: Just watched this amazing do...,1,1
9,#BeforeTheFlood Watch #BeforeTheFlood right he...,1,1


## 2. stratified 80/10/10 split

In [4]:
# split dataset into train, val, test with stratified sampling for later fine-tuning bertweet
# use 80% for training, 10% for validation, and 10% for testing
# reference: https://scikit-learn.org/stable/modules/generated/sklearn.model_selection.train_test_split.html
# code assisted by codex 5.5
from sklearn.model_selection import train_test_split

RANDOM_STATE = 42

train_df, temp_df = train_test_split(
    df,
    test_size=0.20,
    random_state=RANDOM_STATE,
    stratify=df["label"],
)

val_df, test_df = train_test_split(
    temp_df,
    test_size=0.50,
    random_state=RANDOM_STATE,
    stratify=temp_df["label"],
)

train_df = train_df.reset_index(drop=True)
val_df = val_df.reset_index(drop=True)
test_df = test_df.reset_index(drop=True)

print("train:", train_df.shape)
print("val:", val_df.shape)
print("test:", test_df.shape)

pd.DataFrame({
    "train": train_df["label"].value_counts(normalize=True).sort_index(),
    "val": val_df["label"].value_counts(normalize=True).sort_index(),
    "test": test_df["label"].value_counts(normalize=True).sort_index(),
}).rename(index=id2label)

train: (21561, 4)
val: (2695, 4)
test: (2696, 4)


,train,val,test
label,,,
Anti,0.148045,0.148052,0.147997
Pro,0.851955,0.851948,0.852003


In [5]:
# save split files to disk for later use in fine-tuning bertweet
# code assisted by codex 5.5
from pathlib import Path

split_dir = Path("splits")
split_dir.mkdir(exist_ok=True)

train_df.to_csv(split_dir / "train.csv", index=False)
val_df.to_csv(split_dir / "val.csv", index=False)
test_df.to_csv(split_dir / "test.csv", index=False)

print("saved split files to:", split_dir.resolve())

saved split files to: C:\My Documents\CS5664\FinalProject\splits


## 3. load BERTweet tokenizer

In [6]:
# download bertweet tokenizer and test it on an example tweet
# reference: https://huggingface.co/vinai/bertweet-base
# reference: https://huggingface.co/docs/transformers/model_doc/bertweet
# reference: transformers documentation for tokenizers: https://huggingface.co/docs/transformers/main_classes/tokenizer
# code assisted by codex 5.5
from transformers import AutoTokenizer

MODEL_NAME = "vinai/bertweet-base"
MAX_LENGTH = 128

tokenizer = AutoTokenizer.from_pretrained(
    MODEL_NAME,
    normalization=True,
    use_fast=False,
)

example = tokenizer(
    train_df.loc[0, "message"],
    truncation=True,
    padding="max_length",
    max_length=MAX_LENGTH,
)

print(example.keys())
print("input length:", len(example["input_ids"]))

c:\Users\yuesh\miniconda3\envs\hw1\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


KeysView({'input_ids': [0, 246, 5, 22, 250, 4671, 43738, 288, 144, 2594, 144, 552, 2450, 3048, 3921, 31905, 22, 10, 156, 5, 47378, 4235, 30626, 31905, 2, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1], 'attention_mask': [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]})
input length: 128


In [7]:
# make the split datasets into PyTorch Dataset for later use in fine-tuning bertweet
# code assisted by codex 5.5
class TweetDataset(torch.utils.data.Dataset):
    def __init__(self, texts, labels, tokenizer, max_length):
        self.texts = list(texts)
        self.labels = list(labels)
        self.tokenizer = tokenizer
        self.max_length = max_length

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        encoding = self.tokenizer(
            self.texts[idx],
            truncation=True,
            padding="max_length",
            max_length=self.max_length,
            return_tensors="pt",
        )
        item = {key: value.squeeze(0) for key, value in encoding.items()}
        item["labels"] = torch.tensor(self.labels[idx], dtype=torch.long)
        return item


train_dataset = TweetDataset(train_df["message"], train_df["label"], tokenizer, MAX_LENGTH)
val_dataset = TweetDataset(val_df["message"], val_df["label"], tokenizer, MAX_LENGTH)
test_dataset = TweetDataset(test_df["message"], test_df["label"], tokenizer, MAX_LENGTH)

train_dataset[0].keys()

dict_keys(['input_ids', 'attention_mask', 'labels'])

In [8]:
# extract frozen BERTweet embeddings and train a balanced logistic regression classifier
# this gives a closer comparison to frozen TwHIN-BERT + balanced logistic regression
# code assisted by codex 5.5
import torch
import numpy as np

from torch.utils.data import DataLoader
from transformers import AutoModel
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score, f1_score, precision_recall_fscore_support

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

bertweet_encoder = AutoModel.from_pretrained(MODEL_NAME).to(device)
bertweet_encoder.eval()

def extract_cls_embeddings(texts, batch_size=64):
    embeddings = []

    for start in range(0, len(texts), batch_size):
        batch_texts = list(texts.iloc[start:start + batch_size])

        encoded = tokenizer(
            batch_texts,
            truncation=True,
            padding=True,
            max_length=MAX_LENGTH,
            return_tensors="pt",
        )

        encoded = {key: value.to(device) for key, value in encoded.items()}

        with torch.no_grad():
            outputs = bertweet_encoder(**encoded)

        cls_embeddings = outputs.last_hidden_state[:, 0, :]
        embeddings.append(cls_embeddings.cpu().numpy())

    return np.vstack(embeddings)

X_train = extract_cls_embeddings(train_df["message"])
X_val = extract_cls_embeddings(val_df["message"])
X_test = extract_cls_embeddings(test_df["message"])

y_train = train_df["label"].to_numpy()
y_val = val_df["label"].to_numpy()
y_test = test_df["label"].to_numpy()

bertweet_logreg = LogisticRegression(
    class_weight="balanced",
    max_iter=1000,
    random_state=RANDOM_STATE,
)

bertweet_logreg.fit(X_train, y_train)

val_preds = bertweet_logreg.predict(X_val)
test_preds_logreg = bertweet_logreg.predict(X_test)

print("Validation macro-F1:", f1_score(y_val, val_preds, average="macro"))
print("Test results:")
print(classification_report(
    y_test,
    test_preds_logreg,
    target_names=["Anti (-1)", "Pro (1)"],
    digits=4,
    zero_division=0,
))

display(pd.DataFrame(
    confusion_matrix(y_test, test_preds_logreg),
    index=["true Anti", "true Pro"],
    columns=["pred Anti", "pred Pro"],
))

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 49673.66it/s]
RobertaModel LOAD REPORT from: vinai/bertweet-base
Key                             | Status     |  | 
--------------------------------+------------+--+-
lm_head.dense.bias              | UNEXPECTED |  | 
lm_head.layer_norm.bias         | UNEXPECTED |  | 
lm_head.layer_norm.weight       | UNEXPECTED |  | 
lm_head.dense.weight            | UNEXPECTED |  | 
lm_head.bias                    | UNEXPECTED |  | 
lm_head.decoder.weight          | UNEXPECTED |  | 
roberta.embeddings.position_ids | UNEXPECTED |  | 
lm_head.decoder.bias            | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Validation macro-F1: 0.7557206397516163
Test results:
              precision    recall  f1-score   support

   Anti (-1)     0.4934    0.8496    0.6243       399
     Pro (1)     0.9701    0.8485    0.9052      2297

    accuracy                         0.8487      2696
   macro avg     0.7318    0.8491    0.7648      2696
weighted avg     0.8996    0.8487    0.8637      2696



,pred Anti,pred Pro
true Anti,339,60
true Pro,348,1949


## 4. setup metrics

In [9]:
# define metrics for evaluation for bertweet
# reference: sklearn documentation for precision_recall_fscore_support: https://scikit-learn.org/stable/modules/generated/sklearn.metrics.precision_recall_fscore_support.html
# code assisted by codex 5.5
import numpy as np
from sklearn.metrics import accuracy_score, precision_recall_fscore_support


def compute_metrics(eval_pred):
    logits = eval_pred.predictions
    labels = eval_pred.label_ids
    preds = np.argmax(logits, axis=-1)

    precision, recall, f1, _ = precision_recall_fscore_support(
        labels,
        preds,
        average="macro",
        zero_division=0,
    )
    accuracy = accuracy_score(labels, preds)

    return {
        "accuracy": accuracy,
        "macro_f1": f1,
        "macro_precision": precision,
        "macro_recall": recall,
    }

## 5. training methods

In [10]:
# define function to fine-tune bertweet with Trainer API
# there are two runs: one with the backbone frozen and one with the backbone unfrozen
# the frozen backbone only trains the classification head, while the unfrozen backbone trains the entire model
# reference: https://huggingface.co/docs/transformers/trainer
# reference: https://huggingface.co/docs/transformers/model_doc/auto#transformers.AutoModelForSequenceClassification
# code assisted by codex 5.5
from transformers import AutoModelForSequenceClassification, Trainer, TrainingArguments, set_seed
from transformers.utils.notebook import NotebookProgressCallback


def train_bertweet(run_name, freeze_backbone):
    set_seed(RANDOM_STATE)

    model = AutoModelForSequenceClassification.from_pretrained(
        MODEL_NAME,
        num_labels=2,
        id2label=id2label,
        label2id=label2id,
    )

    if freeze_backbone:
        for parameter in model.base_model.parameters():
            parameter.requires_grad = False

    trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
    total_params = sum(p.numel() for p in model.parameters())
    print(f"{run_name}: trainable parameters = {trainable_params:,} / {total_params:,}")

    args = TrainingArguments(
        output_dir=f"outputs/{run_name}",
        per_device_train_batch_size=32,
        per_device_eval_batch_size=64,
        num_train_epochs=3,
        learning_rate=2e-5,
        weight_decay=0.01,
        eval_strategy="epoch",
        save_strategy="epoch",
        load_best_model_at_end=True,
        metric_for_best_model="macro_f1",
        greater_is_better=True,
        save_total_limit=1,
        fp16=True,
        optim="adamw_torch",
        logging_steps=50,
        report_to="none",
        seed=RANDOM_STATE,
        data_seed=RANDOM_STATE,
    )

    trainer = Trainer(
        model=model,
        args=args,
        train_dataset=train_dataset,
        eval_dataset=val_dataset,
        processing_class=tokenizer,
        compute_metrics=compute_metrics,
    )

    trainer.train()
    trainer.remove_callback(NotebookProgressCallback)
    return trainer

## 6. BERTweet training without full fine-tuning (classification head only)

In [11]:
# train bertweet with frozen backbone
# code assisted by codex 5.5
frozen_trainer = train_bertweet(
    run_name="bertweet_frozen_head",
    freeze_backbone=True,
)

frozen_trainer.evaluate()

Loading weights: 100%|██████████| 197/197 [00:00<00:00, 32808.33it/s]
RobertaForSequenceClassification LOAD REPORT from: vinai/bertweet-base
Key                             | Status     | 
--------------------------------+------------+-
lm_head.dense.bias              | UNEXPECTED | 
lm_head.layer_norm.bias         | UNEXPECTED | 
lm_head.layer_norm.weight       | UNEXPECTED | 
lm_head.dense.weight            | UNEXPECTED | 
lm_head.bias                    | UNEXPECTED | 
lm_head.decoder.weight          | UNEXPECTED | 
roberta.pooler.dense.bias       | UNEXPECTED | 
roberta.pooler.dense.weight     | UNEXPECTED | 
roberta.embeddings.position_ids | UNEXPECTED | 
lm_head.decoder.bias            | UNEXPECTED | 
classifier.out_proj.bias        | MISSING    | 
classifier.dense.bias           | MISSING    | 
classifier.dense.weight         | MISSING    | 
classifier.out_proj.weight      | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok 

bertweet_frozen_head: trainable parameters = 592,130 / 134,901,506


Epoch,Training Loss,Validation Loss,Accuracy,Macro F1,Macro Precision,Macro Recall
1,0.394135,0.387895,0.851948,0.460028,0.425974,0.500000
2,0.343366,0.369223,0.851948,0.460028,0.425974,0.500000
3,0.371628,0.363923,0.852319,0.462620,0.926132,0.501253


Writing model shards: 100%|██████████| 1/1 [00:00<00:00,  2.87it/s]
There were missing keys in the checkpoint model loaded: ['roberta.embeddings.LayerNorm.weight', 'roberta.embeddings.LayerNorm.bias', 'roberta.encoder.layer.0.attention.output.LayerNorm.weight', 'roberta.encoder.layer.0.attention.output.LayerNorm.bias', 'roberta.encoder.layer.0.output.LayerNorm.weight', 'roberta.encoder.layer.0.output.LayerNorm.bias', 'roberta.encoder.layer.1.attention.output.LayerNorm.weight', 'roberta.encoder.layer.1.attention.output.LayerNorm.bias', 'roberta.encoder.layer.1.output.LayerNorm.weight', 'roberta.encoder.layer.1.output.LayerNorm.bias', 'roberta.encoder.layer.2.attention.output.LayerNorm.weight', 'roberta.encoder.layer.2.attention.output.LayerNorm.bias', 'roberta.encoder.layer.2.output.LayerNorm.weight', 'roberta.encoder.layer.2.output.LayerNorm.bias', 'roberta.encoder.layer.3.attention.output.LayerNorm.weight', 'roberta.encoder.layer.3.attention.output.LayerNorm.bias', 'roberta.encoder.la

{'eval_loss': 0.36392343044281006,
 'eval_accuracy': 0.8523191094619667,
 'eval_macro_f1': 0.46262024048096195,
 'eval_macro_precision': 0.9261321455085375,
 'eval_macro_recall': 0.5012531328320802,
 'eval_runtime': 2.5563,
 'eval_samples_per_second': 1054.24,
 'eval_steps_per_second': 16.821,
 'epoch': 3.0}

## 7. BERTweet training with full fine-tuning

In [12]:
# train bertweet with unfrozen backbone
# code assisted by codex 5.5
finetuned_trainer = train_bertweet(
    run_name="bertweet_full_finetune",
    freeze_backbone=False,
)

finetuned_trainer.evaluate()

Loading weights: 100%|██████████| 197/197 [00:00<00:00, 49189.06it/s]
RobertaForSequenceClassification LOAD REPORT from: vinai/bertweet-base
Key                             | Status     | 
--------------------------------+------------+-
lm_head.dense.bias              | UNEXPECTED | 
lm_head.layer_norm.bias         | UNEXPECTED | 
lm_head.layer_norm.weight       | UNEXPECTED | 
lm_head.dense.weight            | UNEXPECTED | 
lm_head.bias                    | UNEXPECTED | 
lm_head.decoder.weight          | UNEXPECTED | 
roberta.pooler.dense.bias       | UNEXPECTED | 
roberta.pooler.dense.weight     | UNEXPECTED | 
roberta.embeddings.position_ids | UNEXPECTED | 
lm_head.decoder.bias            | UNEXPECTED | 
classifier.out_proj.bias        | MISSING    | 
classifier.dense.bias           | MISSING    | 
classifier.dense.weight         | MISSING    | 
classifier.out_proj.weight      | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok 

bertweet_full_finetune: trainable parameters = 134,901,506 / 134,901,506


Epoch,Training Loss,Validation Loss,Accuracy,Macro F1,Macro Precision,Macro Recall
1,0.171127,0.153270,0.944341,0.887577,0.895235,0.880364
2,0.130619,0.170792,0.944712,0.879007,0.926665,0.844344
3,0.063290,0.186156,0.951020,0.897776,0.920973,0.878072


Writing model shards: 100%|██████████| 1/1 [00:00<00:00,  2.66it/s]
There were missing keys in the checkpoint model loaded: ['roberta.embeddings.LayerNorm.weight', 'roberta.embeddings.LayerNorm.bias', 'roberta.encoder.layer.0.attention.output.LayerNorm.weight', 'roberta.encoder.layer.0.attention.output.LayerNorm.bias', 'roberta.encoder.layer.0.output.LayerNorm.weight', 'roberta.encoder.layer.0.output.LayerNorm.bias', 'roberta.encoder.layer.1.attention.output.LayerNorm.weight', 'roberta.encoder.layer.1.attention.output.LayerNorm.bias', 'roberta.encoder.layer.1.output.LayerNorm.weight', 'roberta.encoder.layer.1.output.LayerNorm.bias', 'roberta.encoder.layer.2.attention.output.LayerNorm.weight', 'roberta.encoder.layer.2.attention.output.LayerNorm.bias', 'roberta.encoder.layer.2.output.LayerNorm.weight', 'roberta.encoder.layer.2.output.LayerNorm.bias', 'roberta.encoder.layer.3.attention.output.LayerNorm.weight', 'roberta.encoder.layer.3.attention.output.LayerNorm.bias', 'roberta.encoder.la

{'eval_loss': 0.1861564815044403,
 'eval_accuracy': 0.9510204081632653,
 'eval_macro_f1': 0.8977758620689655,
 'eval_macro_precision': 0.9209729879525101,
 'eval_macro_recall': 0.8780717036493673,
 'eval_runtime': 2.5583,
 'eval_samples_per_second': 1053.425,
 'eval_steps_per_second': 16.808,
 'epoch': 3.0}

## 8 Performance

In [13]:
# test both models on the test set and compare results
# code assisted by codex 5.5
test_rows = []

for model_name, trainer in [
    ("BERTweet frozen head", frozen_trainer),
    ("BERTweet full fine-tune", finetuned_trainer),
]:
    metrics = trainer.evaluate(test_dataset)
    row = {"model": model_name}
    row.update({key.replace("eval_", ""): value for key, value in metrics.items()})
    test_rows.append(row)

results_df = pd.DataFrame(test_rows)
results_df

,model,loss,accuracy,macro_f1,macro_precision,macro_recall,runtime,samples_per_second,steps_per_second,epoch
0,BERTweet frozen head,0.358215,0.852003,0.460044,0.426001,0.500000,2.5612,1052.620,16.789,3.0
1,BERTweet full fine-tune,0.191467,0.949184,0.894499,0.914758,0.876987,2.6420,1020.452,16.276,3.0


In [14]:
# print classification report and confusion matrix for both BERTweet models
# code assisted by codex 5.5
from transformers import AutoModelForSequenceClassification, Trainer, TrainingArguments
from sklearn.metrics import classification_report, confusion_matrix
import numpy as np
import pandas as pd

eval_args = TrainingArguments(
    output_dir="tmp_eval",
    per_device_eval_batch_size=64,
    report_to="none",
)

def evaluate_saved_model(model_name, model_path):
    model = AutoModelForSequenceClassification.from_pretrained(
        model_path,
        num_labels=2,
        id2label=id2label,
        label2id=label2id,
    )

    trainer = Trainer(
        model=model,
        args=eval_args,
        eval_dataset=test_dataset,
        processing_class=tokenizer,
    )

    pred_output = trainer.predict(test_dataset)
    preds = np.argmax(pred_output.predictions, axis=-1)
    labels = test_df["label"].to_numpy()

    print(model_name)
    print(classification_report(
        labels,
        preds,
        target_names=["Anti (-1)", "Pro (1)"],
        digits=4,
        zero_division=0,
    ))

    cm = confusion_matrix(labels, preds)
    display(pd.DataFrame(
        cm,
        index=["true Anti", "true Pro"],
        columns=["pred Anti", "pred Pro"],
    ))

    return preds

frozen_preds = evaluate_saved_model(
    "BERTweet frozen head",
    "outputs/bertweet_frozen_head/checkpoint-2022",
)

finetuned_preds = evaluate_saved_model(
    "BERTweet full fine-tune",
    "outputs/bertweet_full_finetune/checkpoint-2022",
)

Loading weights: 100%|██████████| 201/201 [00:00<00:00, 10049.65it/s]


BERTweet frozen head
              precision    recall  f1-score   support

   Anti (-1)     0.0000    0.0000    0.0000       399
     Pro (1)     0.8520    1.0000    0.9201      2297

    accuracy                         0.8520      2696
   macro avg     0.4260    0.5000    0.4600      2696
weighted avg     0.7259    0.8520    0.7839      2696



,pred Anti,pred Pro
true Anti,0,399
true Pro,0,2297


Loading weights: 100%|██████████| 201/201 [00:00<00:00, 9801.48it/s]


BERTweet full fine-tune
              precision    recall  f1-score   support

   Anti (-1)     0.8680    0.7744    0.8185       399
     Pro (1)     0.9615    0.9795    0.9705      2297

    accuracy                         0.9492      2696
   macro avg     0.9148    0.8770    0.8945      2696
weighted avg     0.9477    0.9492    0.9480      2696



,pred Anti,pred Pro
true Anti,309,90
true Pro,47,2250


## 9. save the fine-tuned model

In [15]:
# save the finetuned model, tokenizer, test predictions, and test metrics to disk
# code assisted by codex 5.5
import json
from sklearn.metrics import accuracy_score, precision_recall_fscore_support

final_model_dir = Path("bertweet_pro_anti_final_model")
finetuned_trainer.save_model(final_model_dir)
tokenizer.save_pretrained(final_model_dir)

prediction_df = test_df[["tweetid", "message", "sentiment", "label"]].copy()
prediction_df["pred_label"] = finetuned_preds
prediction_df["pred_sentiment"] = prediction_df["pred_label"].map({0: -1, 1: 1})
prediction_df["pred_name"] = prediction_df["pred_label"].map(id2label)
prediction_df.to_csv("bertweet_test_predictions.csv", index=False)

test_labels = test_df["label"].to_numpy()
macro_precision, macro_recall, macro_f1, _ = precision_recall_fscore_support(
    test_labels,
    finetuned_preds,
    average="macro",
    zero_division=0,
)
weighted_precision, weighted_recall, weighted_f1, _ = precision_recall_fscore_support(
    test_labels,
    finetuned_preds,
    average="weighted",
    zero_division=0,
)
final_metrics = {
    "test_accuracy": float(accuracy_score(test_labels, finetuned_preds)),
    "test_macro_precision": float(macro_precision),
    "test_macro_recall": float(macro_recall),
    "test_macro_f1": float(macro_f1),
    "test_weighted_precision": float(weighted_precision),
    "test_weighted_recall": float(weighted_recall),
    "test_weighted_f1": float(weighted_f1),
}
Path("bertweet_test_metrics.json").write_text(
    json.dumps(final_metrics, indent=2),
    encoding="utf-8",
)

print("saved model to:", final_model_dir.resolve())
print("saved predictions to: bertweet_test_predictions.csv")
print("saved metrics to: bertweet_test_metrics.json")

Writing model shards: 100%|██████████| 1/1 [00:00<00:00,  2.93it/s]

saved model to: C:\My Documents\CS5664\FinalProject\bertweet_pro_anti_final_model
saved predictions to: bertweet_test_predictions.csv
saved metrics to: bertweet_test_metrics.json
